# EEG_29 — Subject-Mean Subtraction: aumentare ε²(parola)?

**Idea**: ε²(soggetto)=0.85 domina il segnale. Se per ogni soggetto calcoliamo
la media di *tutti* i suoi trial (indipendente dalla parola) e la sottraiamo,
rimuoviamo la componente 'firma soggetto' prima del modello.

```
μ_s = mean_{t} x_{s,t}         # (61, 384) — firma soggetto s
x̃_{s,t} = x_{s,t} - μ_s       # residuo parola-dipendente
```

**Problema S-Indep**: il soggetto di test è unseen → non abbiamo μ_s a priori.
Soluzione: calcoliamo μ_s sui trial di test del soggetto (senza usare le label).
La media di tutti i trial non dipende dalla label → nessun data leakage.

**Test rapido**: subset ridotto per velocità.
- TRAIN: 20 soggetti (invece di 50)
- VAL:   5 soggetti
- TEST:  5 soggetti
- 3 seed per stabilità
- Confronto DHSLP con SMS vs senza SMS (baseline)

In [ ]:
import json, logging, re
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg29')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)

# ---- CONFIG ----
N_CHANNELS = 61
N_SAMPLES  = 384
N_CLASSES  = 4
DATA_METRIC = 'abs_pcc'

# Subset ridotto per test rapido
SUBJ_TRAIN = list(range(0,  20))   # 20 soggetti (invece di 50)
SUBJ_VAL   = list(range(50, 55))   # 5 soggetti
SUBJ_TEST  = list(range(60, 65))   # 5 soggetti

# DHSLP (best config da EEG_13)
K_WINDOWS = 8; T_WIN = N_SAMPLES // K_WINDOWS
N_EDGES   = 16; D_MODEL = 64; HIDDEN = 128; N_LAYERS = 2; DROPOUT = 0.3

LR = 1e-3; BATCH_SIZE = 64; MAX_EPOCHS = 50; PATIENCE = 10
LABEL_SMOOTHING = 0.1
SEEDS = [42, 123, 7]

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
label2cluster = {int(k): int(v) for k,v in json.loads(
    (project_root/'configs'/'label_schemes'/'labelid2cluster_concr4.json').read_text()).items()}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log.info(f'device: {device}')
log.info(f'TRAIN={len(SUBJ_TRAIN)} VAL={len(SUBJ_VAL)} TEST={len(SUBJ_TEST)} SEEDS={SEEDS}')

## §2 — Pre-calcolo μ per soggetto

Carica tutti i trial di ogni soggetto e calcola la media (61, 384).
Fatto una volta sola, poi riusato nei dataset.

In [ ]:
def compute_subject_means(subj_ids, metric=DATA_METRIC):
    """
    Calcola μ_s = mean_{trial} x_{s,trial} per ogni soggetto.
    Restituisce dict: sid → tensor (61, 384)
    """
    root = project_root / 'data' / f'hypergraphs_pruned_{metric}'
    accum = defaultdict(list)  # sid → lista di tensor (61,384)
    for p in sorted(root.rglob('trial_*.pt')):
        m = _PAT.match(p.parent.name)
        if not m: continue
        sid = int(m.group(1))
        if sid not in subj_ids: continue
        d = torch.load(p, weights_only=False)
        x = d['x'].float()
        # instance norm prima di accumulare (come nel training)
        x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        accum[sid].append(x)
    means = {}
    for sid, trials in accum.items():
        means[sid] = torch.stack(trials).mean(0)  # (61, 384)
    log.info(f'μ calcolate per {len(means)} soggetti  '
             f'(media trial per soggetto: {np.mean([len(v) for v in accum.values()]):.0f})')
    return means


all_subj = SUBJ_TRAIN + SUBJ_VAL + SUBJ_TEST
log.info('Calcolo μ per soggetto...')
subj_means = compute_subject_means(all_subj)

# Sanity check: la media di un soggetto ha senso?
sid_ex = SUBJ_TRAIN[0]
mu_ex  = subj_means[sid_ex]
log.info(f'P{sid_ex:03d}: μ shape={mu_ex.shape}  |μ|={mu_ex.abs().mean():.4f}')

## §3 — Dataset (con e senza SMS)

`use_sms=True`: applica `x̃ = x - μ_s` dopo l'instance norm.  
`use_sms=False`: solo instance norm — identico a EEG_13 (baseline).

In [ ]:
class EEGDataset(Dataset):
    def __init__(self, subj_ids, subject_means=None, use_sms=False, metric=DATA_METRIC):
        """
        subject_means: dict sid → tensor (61,384). Richiesto se use_sms=True.
        use_sms: applica Subject-Mean Subtraction.
        """
        root = project_root / 'data' / f'hypergraphs_pruned_{metric}'
        self.paths, self.labels, self.sids = [], [], []
        self.subject_means = subject_means
        self.use_sms = use_sms
        for p in sorted(root.rglob('trial_*.pt')):
            m = _PAT.match(p.parent.name)
            if not m: continue
            sid = int(m.group(1))
            if sid not in subj_ids: continue
            d = torch.load(p, weights_only=False)
            y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
            c = label2cluster.get(y_word)
            if c is None: continue
            self.paths.append(p); self.labels.append(c); self.sids.append(sid)
        sms_tag = 'SMS' if use_sms else 'no-SMS'
        log.info(f'  [{sms_tag}] {len(self.paths)} trial | {len(subj_ids)} soggetti')

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        d   = torch.load(self.paths[idx], weights_only=False)
        x   = d['x'].float()
        sid = self.sids[idx]
        # Instance norm (sempre)
        x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        # Subject-Mean Subtraction (opzionale)
        if self.use_sms and self.subject_means is not None and sid in self.subject_means:
            x = x - self.subject_means[sid]
        return x, torch.tensor(self.labels[idx], dtype=torch.long)


def make_loaders(use_sms):
    tr = EEGDataset(SUBJ_TRAIN, subj_means, use_sms=use_sms)
    va = EEGDataset(SUBJ_VAL,   subj_means, use_sms=use_sms)
    te = EEGDataset(SUBJ_TEST,  subj_means, use_sms=use_sms)
    labels   = np.array(tr.labels)
    counts   = np.bincount(labels, minlength=N_CLASSES)
    sample_w = torch.tensor(1.0 / counts[labels], dtype=torch.float)
    sampler  = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)
    kw = dict(num_workers=2, pin_memory=True)
    return (DataLoader(tr, BATCH_SIZE, sampler=sampler, **kw),
            DataLoader(va, BATCH_SIZE, shuffle=False, **kw),
            DataLoader(te, BATCH_SIZE, shuffle=False, **kw))

## §4 — Modello DHSLP (identico a EEG_13, best config)

In [ ]:
class HGNNConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_ch, out_ch))
        self.bias   = nn.Parameter(torch.zeros(out_ch))
        nn.init.xavier_uniform_(self.weight)
    def forward(self, X, H):
        d_v = H.sum(2).clamp(min=1e-6); d_e = H.sum(1).clamp(min=1e-6)
        Dv  = (1.0/d_v.sqrt()).unsqueeze(-1); De = (1.0/d_e).unsqueeze(1)
        out = Dv*(X@self.weight)
        out = torch.bmm(H.transpose(1,2), out)
        out = De.transpose(1,2)*out
        out = torch.bmm(H, out)
        return Dv*out + self.bias

class DHSLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.E       = nn.Parameter(torch.randn(N_EDGES, D_MODEL)*0.01)
        self.pos_enc = nn.Parameter(torch.randn(N_CHANNELS, D_MODEL)*0.01)
        self.node_proj = nn.Sequential(nn.Linear(T_WIN, D_MODEL), nn.LayerNorm(D_MODEL), nn.ELU())
        dims = [D_MODEL]+[HIDDEN]*N_LAYERS
        self.convs = nn.ModuleList([HGNNConv(dims[i], dims[i+1]) for i in range(N_LAYERS)])
        self.bns   = nn.ModuleList([nn.BatchNorm1d(HIDDEN) for _ in range(N_LAYERS)])
        self.drop  = nn.Dropout(DROPOUT)
        self.clf   = nn.Linear(HIDDEN, N_CLASSES)
    def forward(self, x):
        B, N, _ = x.shape; outs = []
        for k in range(K_WINDOWS):
            x_k  = x[:, :, k*T_WIN:(k+1)*T_WIN]
            feat = self.node_proj(x_k) + self.pos_enc
            H_k  = torch.softmax(torch.matmul(feat, self.E.T)/D_MODEL**0.5, dim=2)
            out  = feat
            for conv, bn in zip(self.convs, self.bns):
                out = conv(out, H_k)
                out = bn(out.reshape(B*N,-1)).reshape(B,N,-1)
                out = F.relu(out); out = self.drop(out)
            outs.append(out.mean(1))
        return self.clf(torch.stack(outs,1).mean(1))

assert DHSLP()(torch.randn(2,N_CHANNELS,N_SAMPLES)).shape == (2,N_CLASSES)
log.info('DHSLP OK')

## §5 — Training loop

In [ ]:
def set_seed(s): torch.manual_seed(s); np.random.seed(s)

def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train() if train else model.eval()
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    tot, lbl, pred = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss   = crit(logits, y)
            if train: opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()*len(y)
            lbl.extend(y.cpu().numpy()); pred.extend(logits.argmax(1).cpu().numpy())
    return tot/len(loader.dataset), balanced_accuracy_score(lbl, pred)


def run_experiment(use_sms, seed):
    set_seed(seed)
    tr_l, va_l, te_l = make_loaders(use_sms)
    model = DHSLP().to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
    best_val, best_state, patience_cnt = 0.0, None, 0
    for epoch in range(1, MAX_EPOCHS+1):
        run_epoch(model, tr_l, opt)
        _, va_b = run_epoch(model, va_l)
        sched.step()
        if va_b > best_val:
            best_val  = va_b
            best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= PATIENCE: break
    model.load_state_dict(best_state)
    _, te_b = run_epoch(model, te_l)
    return te_b

## §6 — Confronto SMS vs no-SMS

3 seed × 2 condizioni = 6 run totali (~10 min su GPU).

In [ ]:
results = {'no_sms': [], 'sms': []}

for seed in SEEDS:
    for use_sms, key in [(False, 'no_sms'), (True, 'sms')]:
        tag = 'SMS' if use_sms else 'no-SMS'
        log.info(f'seed={seed} [{tag}]...')
        te_b = run_experiment(use_sms=use_sms, seed=seed)
        results[key].append(te_b)
        log.info(f'  → test bAcc={te_b:.4f}')

m_no  = np.mean(results['no_sms']); s_no  = np.std(results['no_sms'])
m_sms = np.mean(results['sms']);    s_sms = np.std(results['sms'])
delta = m_sms - m_no

print(f'\n========== RISULTATI ({len(SEEDS)} seed) ==========')
print(f'  no-SMS:  {m_no:.4f} ± {s_no:.4f}')
print(f'  SMS:     {m_sms:.4f} ± {s_sms:.4f}')
print(f'  Δ(SMS - no-SMS): {delta:+.4f}')
print(f'  per-seed no-SMS: {[f"{v:.4f}" for v in results["no_sms"]]}')
print(f'  per-seed SMS:    {[f"{v:.4f}" for v in results["sms"]]}')

## §7 — Plot e analisi del segnale

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'EEG_29 — Subject-Mean Subtraction\n'
             f'Subset: {len(SUBJ_TRAIN)} train / {len(SUBJ_VAL)} val / {len(SUBJ_TEST)} test soggetti',
             fontsize=13, fontweight='bold')

# --- Bar chart ---
ax = axes[0]
means  = [m_no, m_sms]
stds   = [s_no, s_sms]
colors = ['#90CAF9', '#A5D6A7']
ax.bar(['no-SMS\n(baseline)', 'SMS'], means, color=colors, alpha=0.9, edgecolor='none')
ax.errorbar([0, 1], means, yerr=stds, fmt='none', color='black', capsize=6, lw=2)
ax.axhline(1/N_CLASSES, color='gray', ls='--', lw=1.5, label='Chance (25%)')
ax.set_ylabel('Test bAcc'); ax.set_title('Test bAcc (media ± std)')
ax.legend(fontsize=9)
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 0.004, f'{m:.3f}\n±{s:.3f}', ha='center', fontsize=10, fontweight='bold')
if delta > 0:
    ax.annotate(f'Δ={delta:+.4f}', xy=(0.5, max(means)+s_sms+0.015),
                ha='center', fontsize=11, color='#2E7D32', fontweight='bold')

# --- Visualizzazione μ soggetto vs trial singolo ---
ax2 = axes[1]
sid_ex = SUBJ_TRAIN[0]
mu_ex  = subj_means[sid_ex].numpy()  # (61, 384)

# Carica un trial di esempio
root_ex = project_root / 'data' / f'hypergraphs_pruned_{DATA_METRIC}'
trial_ex = None
for p in root_ex.rglob('trial_*.pt'):
    m = _PAT.match(p.parent.name)
    if m and int(m.group(1)) == sid_ex:
        d = torch.load(p, weights_only=False)
        x = d['x'].float()
        x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        trial_ex = x.numpy(); break

t = np.linspace(0, 384/256, 384)
CH = 0  # primo canale
if trial_ex is not None:
    ax2.plot(t, trial_ex[CH], color='#90CAF9', alpha=0.8, lw=1, label='Trial singolo')
    ax2.plot(t, mu_ex[CH],   color='#E65100',  lw=2,     label='μ soggetto (componente rimossa)')
    residuo = trial_ex[CH] - mu_ex[CH]
    ax2.plot(t, residuo,     color='#2E7D32',  lw=1.5,   label='Residuo (trial - μ)', ls='--')
    ax2.axhline(0, color='gray', ls=':', lw=0.8)
    ax2.set_xlabel('Tempo (s)'); ax2.set_ylabel('μV (norm)')
    ax2.set_title(f'P{sid_ex:03d} — canale {CH}: trial vs μ vs residuo')
    ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg29_sms_results.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Varianza spiegata: stima rapida ---
# Carica un campione di trial e calcola var(segnale) vs var(residuo)
sample_raw, sample_res = [], []
root_ex = project_root / 'data' / f'hypergraphs_pruned_{DATA_METRIC}'
count = 0
for p in root_ex.rglob('trial_*.pt'):
    m_match = _PAT.match(p.parent.name)
    if not m_match: continue
    sid = int(m_match.group(1))
    if sid not in SUBJ_TRAIN: continue
    d = torch.load(p, weights_only=False)
    x = d['x'].float()
    x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
    sample_raw.append(x.var().item())
    if sid in subj_means:
        sample_res.append((x - subj_means[sid]).var().item())
    count += 1
    if count >= 500: break  # campione di 500 trial

var_raw = np.mean(sample_raw)
var_res = np.mean(sample_res)
print(f'\nVarianza media trial grezzo (post inst-norm): {var_raw:.4f}')
print(f'Varianza media trial residuo (post SMS):      {var_res:.4f}')
print(f'Riduzione varianza: {(1 - var_res/var_raw)*100:.1f}%  '
      f'(= componente soggetto rimossa)')

## §8 — Interpretazione

**Se Δ > 0** (SMS migliora):
- La componente soggetto era effettivamente rumore per il task parola
- Vale la pena applicarlo a EEG_13 full (50 train) e confrontare
- Potrebbe spiegare parte del gap Li et al. vs nostri risultati

**Se Δ ≈ 0** (SMS indifferente):
- Il modello già imparava a ignorare la firma soggetto (instance norm + DHSLP end-to-end)
- Il problema è davvero ε²(parola)=0.03 — non c'è segnale parola da estrarre

**Se Δ < 0** (SMS peggiora):
- La firma soggetto conteneva informazione utile anche per la parola
- Sottraendola rimuoviamo anche segnale parola → controproducente

**Riduzione varianza** (ultima cella §7):
- Quanto % della varianza è la firma soggetto?
- Se ~85% → coerente con ε²(soggetto)=0.85 — la SMS ha rimosso quasi tutto il segnale
- Il residuo è solo il 15% → pochissimo segnale rimasto, di cui 3% è parola